In [ ]:
import polars as pl 
import datetime as dt
import sys
import torch
import torch.nn as nn 
import numpy as np
import yaml


In [ ]:

df_path = "/home/dhruvkumarjiguda/code/Time-Series-Library/test/data/process_metrics_202604131435.csv"
yaml_path = "/home/dhruvkumarjiguda/code/Time-Series-Library/test/data/process_logic.yaml"

# df = pl.read_csv(df_path)    
# df.head(3)


# function definitions

In [ ]:
# class datasetbuilder:
#     def __init__(self, path:str):
        
#         self.df = pl.read_csv(path)


def parse_fsm(yaml_path: str):
    with open(yaml_path) as f:
        fsm_config = yaml.safe_load(f)

    fsms = {}

    for name, fsm in fsm_config["definitions"].items():

        state_to_id = {s: i for i, s in enumerate(fsm["states"].keys())}

        transition_map = {
            (state, t["event"]): t["next_state"]
            for state, state_def in fsm["states"].items()
            for t in state_def.get("transitions", [])
        }

        fsms[name] = (state_to_id, transition_map)

    return fsms

def group_sequences(df: pl.DataFrame) -> list[pl.DataFrame]:
    return [group for _, group in df.group_by("sequence_id")]

def load_data(csv_path: str) -> pl.DataFrame:
    df = pl.read_csv(csv_path)
    
    # drop useless cols
    df = df.drop(["pallet_serial_number","id"])
    
    # convert timestamp col from str to datetime
    df = df.with_columns(
        pl.col("timestamp")
        .str.strptime(
            pl.Datetime,
            format="%Y-%m-%d %H:%M:%S%.f %z"
        )
        # .dt.convert_time_zone("UTC")  # normalize
    )

    # make sequence_id col
    df = df.with_columns(
        pl.concat_str(
            [
                pl.col("timestamp").dt.date().cast(pl.Utf8),
                pl.col("cycle_count").cast(pl.Utf8)
            ],
            separator="_"
        ).alias("sequence_id")
    )
    
    # sort
    # df = df.sort([ "timestamp","sequence_id"])

    return df


def pivot_sequence(seq_df: pl.DataFrame):
    # unique sorted axes
    timestamps = seq_df["timestamp"].unique().sort().to_list()
    metrics = seq_df["metric_name"].unique().sort().to_list()

    # index maps (O(1) lookup)
    t_map = {t: i for i, t in enumerate(timestamps)}
    m_map = {m: i for i, m in enumerate(metrics)}

    T, D = len(timestamps), len(metrics)

    # allocate
    X = np.full((T, D), np.nan)
    M = np.zeros((T, D))

    # fill
    for row in seq_df.iter_rows(named=True):
        t_idx = t_map[row["timestamp"]]
        m_idx = m_map[row["metric_name"]]

        X[t_idx, m_idx] = row["value"]
        M[t_idx, m_idx] = 1

    return timestamps, X, M, m_map


# Step 6: Compute Time Deltas
import numpy as np
def compute_delta_t(timestamps) -> np.ndarray:
    T = len(timestamps)
    delta_t = np.zeros(T)
    for t in range(1, T):
        delta_t[t] = (timestamps[t] - timestamps[t-1]).total_seconds()
    return delta_t

# Step 7: Time Since Last Observation (per feature)
def compute_delta_obs(M: np.ndarray, delta_t: np.ndarray) -> np.ndarray:
    T, D = M.shape
    delta_obs = np.zeros((T, D))
    
    for d in range(D):
        last_time = 0
        for t in range(T):
            if M[t, d] == 1:
                delta_obs[t, d] = 0
                last_time = 0
            else:
                last_time += delta_t[t]
                delta_obs[t, d] = last_time
    return delta_obs

# Step 8: Reconstruct State Timeline (if FSM provided)
def reconstruct_state_timeline(seq_df: pl.DataFrame, state_to_id: dict, transition_map: dict) -> tuple:
    timestamps = seq_df["timestamp"].unique().sort()
    state_ids = []
    transition_ids = []
    time_in_state = np.zeros(len(timestamps))
    
    current_state = "initial"  # from FSM config
    state_start_time = 0
    
    for i, ts in enumerate(timestamps):
        # infer event from row at this timestamp
        # next_state = transition_map.get((current_state, event), current_state)
        
        if i > 0:
            time_in_state[i] = (ts - timestamps[i-1]).total_seconds()
        
        state_ids.append(state_to_id.get(current_state, 0))
        # transition_ids.append(...)
    
    return np.array(state_ids), time_in_state

# Step 9: Gap/Downtime Features
def compute_gap_features(delta_t: np.ndarray, threshold_seconds: float = 300) -> tuple:
    T = len(delta_t)
    is_gap = (delta_t > threshold_seconds).astype(float)
    gap_duration = np.where(is_gap == 1, delta_t, 0)
    return is_gap, gap_duration


def build_dataset(csv_path: str, fsm_yaml_path: str = None, gap_threshold: float = 300):

    # loads the df, and adds sequence_id
    df = load_data(csv_path)
    
    #this needs to change. this currently returns a dict of dict of state_to_id and trans_map
    # state_to_id, transition_map = parse_fsm(fsm_yaml_path) if fsm_yaml_path else ({}, {})
    
    sequences = group_sequences(df)
    dataset = []
    
    for seq_df in sequences:
        timestamps, X, M, metric_map = pivot_sequence(seq_df)
        delta_t = compute_delta_t(timestamps)
        delta_obs = compute_delta_obs(M, delta_t)
        

        
        is_gap, gap_duration = compute_gap_features(delta_t, gap_threshold)
        
        dataset.append({
            "sequence_id": seq_df["sequence_id"][0],
            "timestamps": timestamps,
            "X": X,
            "M": M,
            "delta_t": delta_t,
            "delta_obs": delta_obs,
            "is_gap": is_gap,
            "gap_duration": gap_duration,
        })
    
    return dataset

In [ ]:
# Step 11: Convert to PyTorch Dataset
import torch
from torch.utils.data import Dataset
class ProcessAwareIrregularDataset(Dataset):
    def __init__(self, sequences, window_size=None, horizon=1):
        self.samples = []
        for seq in sequences:
            X = torch.tensor(seq["X"], dtype=torch.float32)
            M = torch.tensor(seq["M"], dtype=torch.float32)
            delta_t = torch.tensor(seq["delta_t"], dtype=torch.float32).unsqueeze(-1)
            delta_obs = torch.tensor(seq["delta_obs"], dtype=torch.float32)
            
            features = [X, M, delta_obs, delta_t]
            if seq["state_id"] is not None:
                features.append(torch.tensor(seq["state_id"], dtype=torch.long).unsqueeze(-1))
            
            features = torch.cat(features, dim=-1)
            
            T = features.shape[0]
            if window_size is None:
                self.samples.append((features, X))
            else:
                for t in range(T - window_size - horizon + 1):
                    self.samples.append((
                        features[t:t+window_size],
                        X[t+window_size:t+window_size+horizon]
                    ))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]
---
## Output Summary
Each sample returns:
- **X**: [T, D] metric values
- **M**: [T, D] mask (1 = observed)
- **delta_t**: [T, 1] time since last timestep
- **delta_obs**: [T, D] time since last observation per feature
- **state_id**: [T, 1] (if FSM provided)
- **time_in_state**: [T] (if FSM provided)
---

In [ ]:
dataset = build_dataset(csv_path=df_path,fsm_yaml_path=yaml_path)



# tests

In [ ]:
out = load_data(df_path)

lists = group_sequences(out)

first_sequence = lists[0]


# wide_table = pivot_sequence(first_sequence)

# print(wide_table)

In [ ]:
timestamps, X, M, m_map = wide_table

# reconstruct column order
cols = [None] * len(m_map)
for k, v in m_map.items():
    cols[v] = k

# build wide dataframe
wide_df = (
    pl.DataFrame(X, schema=cols)
    .with_columns(pl.Series("timestamp", timestamps))
    .select(["timestamp"] + cols)
)

# optional: include mask columns for debugging missingness
mask_df = pl.DataFrame(M, schema=[f"{c}_mask" for c in cols])

wide_with_mask = pl.concat(
    [wide_df, mask_df],
    how="horizontal"
)

# display settings (adjust if large)
pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_cols(20)

# show
# print(wide_with_mask)
wide_with_mask.write_csv("/home/dhruvkumarjiguda/code/Time-Series-Library/test/data/test_output.csv")


In [ ]:
# first_sequence.sort("timestamp").filter(
#     pl.col("timestamp").count().over("timestamp") > 1
# ).head(10)

# # first_sequence.filter(
# #     pl.col("timestamp").count().over("timestamp") > 1
# # ).group_by("timestamp").agg(
# #     pl.col("state_context").n_unique().alias("n_states"),
# #     pl.col("state_context").unique().alias("states")
# # ).filter(
# #     pl.col("n_states") > 1
# # )

In [ ]:
values = (
    df.select(
        pl.concat_str(["station_name", "metric_name"], separator="__")
        .alias("station_metric")
    )
    .unique()
    .get_column("station_metric")   # extract Series
    .to_list()                      # convert to Python list
)

with open("metrics.txt", "w") as f:
    f.write("\n".join(values))
    
with open("metrics.txt", "w") as f:
    f.write("\n".join(values))

In [ ]:
df = df.with_columns(
    pl.col("timestamp")
    .str.strptime(
        pl.Datetime,
        format="%Y-%m-%d %H:%M:%S%.f %z"
    )
    # .dt.convert_time_zone("UTC")  # normalize
)



In [ ]:
df.select(
    pl.col("timestamp").min().alias("start"),
    pl.col("timestamp").max().alias("end"),
)